# Corpus Analytics: The Singlet Atlas

This notebook explores the Singlet atlas — a growing corpus of uniformly
processed single-cell datasets from NCBI GEO. All samples are processed
by the singlify pipeline (alignment, quantification, QC) and made queryable
via the `singlet` Python package.

Current corpus: **2,250 samples** across **1,131 series** and **7 species**.

In [1]:
import singlet
import pandas as pd
import numpy as np

# Overview
singlet.summary()

singlet atlas: 2,250 samples (924 SUCCESS) • 1,131 series • 10 species • 2.7M cells


'singlet atlas: 2,250 samples (924 SUCCESS) • 1,131 series • 10 species • 2.7M cells'

In [2]:
# Species breakdown
singlet.species()

['Danio rerio',
 'Drosophila melanogaster',
 'Gallus gallus',
 'Homo sapiens',
 'Macaca mulatta',
 'Mus musculus',
 'Pan troglodytes']

In [3]:
# Dataset overview — top series by sample count
singlet.top_series(n=15)

,gse_id,n_samples,total_cells,avg_mapping_rate,avg_median_genes,organism
0,GSE174399,8,118126.0,0.957237,243.000000,Homo sapiens
1,GSE127918,12,116925.0,0.661725,175.250000,Homo sapiens
2,GSE173756,12,102346.0,0.904283,413.750000,Homo sapiens
3,GSE245310,13,86718.0,0.929429,107.307692,Homo sapiens
4,GSE200277,8,53318.0,0.839870,525.250000,Homo sapiens
5,GSE176294,15,50312.0,0.978747,866.800000,Homo sapiens
6,GSE81750,27,46145.0,0.683210,270.560000,Homo sapiens
7,GSE264667,3,34145.0,0.908076,593.666667,Homo sapiens
8,GSE81749,15,33713.0,0.777160,340.066667,Homo sapiens
9,GSE281106,26,32912.0,0.933634,1501.259259,Homo sapiens


In [4]:
# Get the full sample catalog for analysis
df = singlet.sample_index()
print(f'Total samples: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(3)

Total samples: 2,250
Columns: ['gsm_id', 'gse_id', 'organism', 'protocol', 'status', 'cells_called', 'mapping_rate', 'median_genes', 'title']


,gsm_id,gse_id,organism,protocol,status,cells_called,mapping_rate,median_genes,title
0,GSM2706111,GSE101561,Homo sapiens,10xv3,SOFT_FAIL,0.0,0.963600,0.0,Akata CAGE 1
1,GSM2406685,GSE90546,Homo sapiens,10xv2,HARD_FAIL,0.0,0.000000,NaN,UPR Perturb-seq experiment guide barcodes (gem...
2,GSM2325892,GSE87239,Homo sapiens,None,SOFT_FAIL,3.0,0.355882,NaN,Exp1_A1


In [5]:
# Processing status breakdown
status_counts = df['status'].value_counts()
print('Processing Status:')
for status, count in status_counts.items():
    print(f'  {status}: {count:,} ({count/len(df):.1%})')

Processing Status:
  SUCCESS: 924 (41.1%)
  HARD_FAIL: 796 (35.4%)
  SOFT_FAIL: 530 (23.6%)


In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Filter to successful samples for QC analysis
success = df[df['status'] == 'SUCCESS'].copy()
print(f'Successful samples: {len(success):,}')

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Mapping rate distribution
if 'mapping_rate' in success.columns:
    mr = success['mapping_rate'].dropna()
    axes[0,0].hist(mr, bins=50, color='#3b82f6', edgecolor='white')
    axes[0,0].axvline(mr.median(), color='red', linestyle='--', label=f'median={mr.median():.1%}')
    axes[0,0].set_xlabel('Mapping Rate')
    axes[0,0].set_ylabel('Samples')
    axes[0,0].set_title(f'Mapping Rate (n={len(mr)})')
    axes[0,0].legend()

# 2. Cells per sample
if 'cells_called' in success.columns:
    cells = success['cells_called'].dropna()
    axes[0,1].hist(np.log10(cells+1), bins=50, color='#22c55e', edgecolor='white')
    axes[0,1].axvline(np.log10(cells.median()+1), color='red', linestyle='--', 
                       label=f'median={cells.median():,.0f}')
    axes[0,1].set_xlabel('log10(Cells)')
    axes[0,1].set_ylabel('Samples')
    axes[0,1].set_title(f'Cells Called (n={len(cells)})')
    axes[0,1].legend()

# 3. Median genes per cell
if 'median_genes' in success.columns:
    genes = success['median_genes'].dropna()
    axes[1,0].hist(genes, bins=50, color='#a855f7', edgecolor='white')
    axes[1,0].axvline(genes.median(), color='red', linestyle='--',
                       label=f'median={genes.median():,.0f}')
    axes[1,0].set_xlabel('Median Genes/Cell')
    axes[1,0].set_ylabel('Samples')
    axes[1,0].set_title(f'Median Genes per Cell (n={len(genes)})')
    axes[1,0].legend()

# 4. Species pie chart
if 'organism' in success.columns:
    org_counts = success['organism'].value_counts().head(6)
    axes[1,1].pie(org_counts.values, labels=org_counts.index, autopct='%1.0f%%',
                  colors=['#3b82f6', '#22c55e', '#a855f7', '#f59e0b', '#ef4444', '#6b7280'])
    axes[1,1].set_title('Species Distribution (SUCCESS)')

plt.suptitle('Singlet Atlas — Corpus Quality Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('corpus_analytics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: corpus_analytics.png')

Successful samples: 924


Saved: corpus_analytics.png


In [7]:
# Quality metrics summary for successful samples
if 'mapping_rate' in success.columns:
    print('Quality Metrics Summary (SUCCESS samples):')
    print(f'  Mapping Rate:     median={success["mapping_rate"].median():.1%}, '
          f'IQR=[{success["mapping_rate"].quantile(0.25):.1%}, {success["mapping_rate"].quantile(0.75):.1%}]')
if 'cells_called' in success.columns:
    print(f'  Cells Called:     median={success["cells_called"].median():,.0f}, '
          f'IQR=[{success["cells_called"].quantile(0.25):,.0f}, {success["cells_called"].quantile(0.75):,.0f}]')
if 'median_genes' in success.columns:
    print(f'  Median Genes:     median={success["median_genes"].median():,.0f}, '
          f'IQR=[{success["median_genes"].quantile(0.25):,.0f}, {success["median_genes"].quantile(0.75):,.0f}]')
if 'median_umis' in success.columns:
    print(f'  Median UMIs:      median={success["median_umis"].median():,.0f}, '
          f'IQR=[{success["median_umis"].quantile(0.25):,.0f}, {success["median_umis"].quantile(0.75):,.0f}]')

Quality Metrics Summary (SUCCESS samples):
  Mapping Rate:     median=80.4%, IQR=[69.4%, 88.8%]
  Cells Called:     median=1,167, IQR=[339, 3,245]
  Median Genes:     median=195, IQR=[99, 604]


In [8]:
# Find high-quality datasets (many cells, high mapping rate, many genes)
hq = success[
    (success.get('cells_called', pd.Series(dtype=float)) > 5000) &
    (success.get('mapping_rate', pd.Series(dtype=float)) > 0.8) &
    (success.get('median_genes', pd.Series(dtype=float)) > 2000)
].copy() if all(c in success.columns for c in ['cells_called', 'mapping_rate', 'median_genes']) else pd.DataFrame()

if len(hq) > 0:
    print(f'High-quality samples (>5K cells, >80% mapping, >2K genes): {len(hq):,}')
    print(f'That\'s {len(hq)/len(success):.0%} of all successful samples')
    print(f'\nTop 5 by cell count:')
    display(hq.nlargest(5, 'cells_called')[['gsm_id', 'organism', 'cells_called', 'mapping_rate', 'median_genes', 'title']].reset_index(drop=True))
else:
    print('QC columns not available for high-quality filtering')

High-quality samples (>5K cells, >80% mapping, >2K genes): 3
That's 0% of all successful samples

Top 5 by cell count:


,gsm_id,organism,cells_called,mapping_rate,median_genes,title
0,GSM6938432,Homo sapiens,6637.0,0.883400,2546.0,10xscRNAseq_NANOS3+PGCLC
1,GSM6595044,Homo sapiens,5154.0,0.899315,2131.0,organoids_4-month-old-PT
2,GSM7911433,Homo sapiens,5145.0,0.900473,2092.0,"UCB, CRCY, replicate 2"


## Conclusion

The Singlet atlas provides a growing, uniformly-processed corpus of single-cell
data. Key features:

- **Uniform processing**: Every sample goes through the same singlify pipeline
- **Rich QC**: Mapping rate, cell counts, genes/cell, UMIs/cell, doublet rates
- **Multi-species**: Human, mouse, rat, and more
- **Queryable**: `pip install singlet-bio` gives instant access to the catalog

The `singlet` package makes it easy to browse, filter, and load data from this
atlas for downstream analysis.